# 01 — Data Collection

**Project:** Private and Public Good Provision of Mental Health Services in France  
**Authors:** Salma El Aazdoudi, Cynthia Francis, Anahi Reyes Miguel, Rafaël Mourouvin  
**Course:** Data Analysis in Economics — Telecom Paris

---

## Overview

This notebook collects therapist data from [Psychologue.net](https://www.psychologue.net), a French online directory of licensed psychologists and therapists. Two complementary scraping strategies are implemented:

- **Method 1 (Selenium + BeautifulSoup):** Automates a browser to progressively load all therapist cards on the search page, then parses the resulting HTML snapshot to extract structured JSON-LD data embedded in `<script>` tags.
- **Method 2 (Hidden API):** Bypasses the front-end entirely by querying an undocumented internal REST API discovered via browser DevTools, retrieving paginated therapist records in JSON format along with real-time appointment availability.

Data was collected on **two dates** (December 23, 2024 and January 13, 2025) to capture scheduling availability across time. The final output is a merged CSV file (`therapists_final.csv`) used in all downstream analyses.

---

## Required Libraries

In [ ]:
import time
import json
import csv
import requests
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException

---

## Method 1: Selenium + BeautifulSoup

Psychologue.net uses a dynamic single-page application (SPA) that loads therapist cards lazily behind a "Voir plus de cabinets" button. Traditional static scraping cannot capture the full listing — we use **Selenium** to automate browser interactions and load all results before scraping.

The workflow is split into two sub-steps:
1. **Save a full HTML snapshot** by clicking the load-more button repeatedly until all therapists appear.
2. **Parse the snapshot** to extract therapist details and appointment slot availability.

### Step 1.1 — Save the HTML Snapshot

A headless Chrome session navigates to the search page and clicks "Voir plus de cabinets" in a loop until the button disappears, indicating all records have been loaded. After each click the full page source is saved to `webpage_snapshot.html`.  
> ⚠️ **Run this cell only once** — it takes several minutes to complete.

In [ ]:
import os

# Configure Selenium WebDriver
chrome_options = Options()
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--incognito")
# chrome_options.add_argument("--headless")  # Uncomment for headless mode

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

url = "https://www.psychologue.net/search?therapytype=0&service=0&location[]=0&neighborhood=null"
driver.get(url)

In [ ]:
def safe_click(driver, button_selector):
    """Click the 'Load More' button safely, restarting the session on failure."""
    try:
        load_more_button = driver.find_element(By.CSS_SELECTOR, button_selector)
        driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", load_more_button)
        time.sleep(2)
    except WebDriverException as e:
        print(f"Error clicking 'Load More': {e}")
        driver.quit()
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=chrome_options
        )
        driver.get(url)
        safe_click(driver, button_selector)
    return driver

In [ ]:
# Repeatedly click "Voir plus de cabinets" until all therapists are loaded
snapshot_path = "data/raw/webpage_snapshot.html"
os.makedirs("data/raw", exist_ok=True)

while True:
    try:
        load_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "see_more_companies"))
        )
        driver.execute_script("arguments[0].click();", load_more_button)
        time.sleep(2)

        with open(snapshot_path, "w", encoding="utf-8") as file:
            file.write(driver.page_source)
        print(f"Snapshot saved to {snapshot_path}")

    except Exception as e:
        print("No more 'Voir plus de cabinets' button or an error occurred:", e)
        break

### Step 1.2 — Parse the HTML Snapshot

The saved snapshot embeds structured therapist data as JSON-LD objects (`@type: Physician`) inside `<script>` tags. We load the snapshot into a fresh browser session to also be able to query dynamic CSS selectors for pricing information, then iterate over all script tags to extract each therapist's name, location, rating, pricing, and appointment availability.

In [ ]:
# Load the saved HTML snapshot into a browser session for parsing
chrome_options = Options()
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--incognito")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

local_file_path = os.path.abspath("data/raw/webpage_snapshot.html")
local_file_url = f"file://{local_file_path}"
driver.get(local_file_url)

In [ ]:
# Parse the loaded page with BeautifulSoup and extract therapist records
soup = BeautifulSoup(driver.page_source, "html.parser")
script_tags = soup.find_all("script", type="application/ld+json")

therapists = []

for idx, script in enumerate(script_tags):
    try:
        data = json.loads(script.string)

        if data.get("@type") == "Physician":
            name         = data.get("name")
            location     = data.get("address", {}).get("addressLocality")
            postal       = data.get("address", {}).get("postalCode")
            rating       = data.get("aggregateRating", {}).get("ratingValue")
            review_count = data.get("aggregateRating", {}).get("reviewCount")

            # Price is rendered dynamically — query it via Selenium
            try:
                price_selector = (
                    f"#company_list_wrapper > article:nth-child({idx + 1}) > "
                    "div.sc-6686e129-1.iGARLG > ul.sc-308fd480-2.kLkEho > "
                    "li.sc-6686e129-26.kZsZiH > p > strong"
                )
                price_element = driver.find_element(By.CSS_SELECTOR, price_selector)
                price = price_element.text
            except Exception:
                price = "Not available"

            # Extract appointment availability from the rendered DOM
            availability_section = soup.select_one(
                f"#company_list_wrapper > article:nth-child({idx + 1}) > "
                "div > div.sc-b2fc3406-5.hfDahw > div.sc-6ecf1888-0.WFsmp > "
                "div.sc-6ecf1888-2.cZmjZS > div.sc-6ecf1888-3.fWPnyV"
            )

            datetime_list = []
            if availability_section:
                all_dates = availability_section.select("div.sc-6ecf1888-4.hYyARV")
                day_dates = [
                    (
                        el.select_one("div.sc-6ecf1888-7 > div.sc-6ecf1888-8").text,
                        el.select_one("div.sc-6ecf1888-7 > div.sc-6ecf1888-9").text
                    )
                    for el in all_dates
                ]
                times_array = []
                for day_el in all_dates:
                    times = day_el.select("div.sc-84222868-0 > div.sc-6ecf1888-1")
                    temp = []
                    for elt in times:
                        button = elt.select_one("div > button")
                        if button and "TFFWk" not in button.get("class", []):
                            temp.append(button.text.strip())
                    times_array.append(temp)

                for (day_name, day_date), time_list in zip(day_dates, times_array):
                    for t in time_list:
                        full_dt = f"{day_name} {day_date} {datetime.today().year} {t}"
                        datetime_list.append(full_dt)

            if not datetime_list:
                datetime_list = ["N/A"]

            for dt in datetime_list:
                therapists.append({
                    "Name": name,
                    "Location": location,
                    "Postal": postal,
                    "Rating": rating,
                    "Reviews": review_count,
                    "Price": price,
                    "Availability": dt
                })

    except (json.JSONDecodeError, AttributeError):
        continue

driver.quit()
print(f"Extracted {len(therapists)} records from snapshot.")

### Step 1.3 — Save Method 1 Results to CSV

In [ ]:
os.makedirs("data/raw", exist_ok=True)
csv_file = "data/raw/therapists_method1.csv"
fieldnames = ["Name", "Location", "Postal", "Rating", "Reviews", "Price", "Availability"]

try:
    with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(therapists)
    print(f"Data successfully saved to {csv_file}")
except OSError as e:
    print(f"Error saving CSV file: {e}")

---

## Method 2: Hidden API

While exploring the website's network traffic via browser DevTools, we discovered that the front-end fetches therapist records from an undocumented internal API at `https://www.psychologue.net/companies`. This API returns paginated JSON responses with richer and more reliable structured data than what can be parsed from the HTML.

For each therapist, we additionally query a second endpoint to retrieve real-time calendar availability in 5-day windows. This method is significantly faster and more robust than Selenium-based scraping.

### Step 2.1 — Configure API Headers and Parameters

The API requires specific headers (including a session cookie and `x-project-id`) to respond correctly. These were captured from a live browser session. Pagination runs from `page_start` to `page_end` — adjust the range to control coverage.

In [ ]:
cabinets_url = "https://www.psychologue.net/companies"

headers = {
    "accept": "application/json, text/plain, */*",
    "accept-encoding": "gzip, deflate, br",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "pragma": "no-cache",
    "referer": "https://www.psychologue.net/cabinets",
    "sec-ch-ua": '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "x-project-id": "75",
    # NOTE: Replace the cookie below with a fresh session cookie if requests start failing.
    "cookie": "PHPSESSID=1ac359c2d1aedc9a75e7671f03288bcb"
}

output_file = "data/raw/therapists_api.csv"
page_start  = 1
page_end    = 1200  # Adjust as needed

### Step 2.2 — Fetch Appointment Availability

For each therapist with a registered calendar, we query the scheduling API in sliding 5-day windows to collect available appointment slots. The `repetitions` parameter controls how many 5-day windows to scan.

In [ ]:
def get_availability(calendar_id, start_date, repetitions):
    """
    Fetch available appointment slots for a therapist from the scheduling API.

    Parameters
    ----------
    calendar_id : str
        The therapist's internal calendar identifier.
    start_date : str
        Start date for the availability query, in 'YYYY-MM-DD' format.
    repetitions : int
        Number of 5-day windows to query forward from start_date.

    Returns
    -------
    list of str
        A flat list of available slot start/end timestamps.
    """
    dates_available = []
    current_date = datetime.strptime(start_date, "%Y-%m-%d")

    for _ in range(repetitions):
        date_str = current_date.strftime("%Y-%m-%d")
        endpoint = (
            f"https://www.psychologue.net/api/schedule/v1.0/calendars/"
            f"{calendar_id}/fromDay?day={date_str}&numberOfDays=5&firstAvailable=0"
        )
        response = requests.get(endpoint, headers=headers)

        if response.status_code == 400:
            current_date += timedelta(days=5)
            continue

        try:
            data = response.json()
            for period in data["data"]["freeBusyPeriods"]:
                if period.get("isBusy", "N/A") == 0:
                    dates_available.append(period.get("start", "N/A"))
                    dates_available.append(period.get("end", "N/A"))
        except Exception:
            pass

        current_date += timedelta(days=5)

    return dates_available

### Step 2.3 — Paginate Through the API and Save Results

We iterate over paginated API responses and write each therapist record (one row per availability slot) to a CSV file. Pages returning a 400 status code are skipped. The loop terminates when the `results` field is empty, indicating no further records.

In [ ]:
os.makedirs("data/raw", exist_ok=True)

with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Name", "Location", "Postal Code", "Rating", "Price", "Availability", "Services"])

    page = page_start
    while page <= page_end:
        params = {
            "therapytype": 0,
            "pricefilter": 0,
            "reviewfilter": 0,
            "page": page,
            "orderby": "position",
        }
        response = requests.get(cabinets_url, headers=headers, params=params)

        if response.status_code == 400:
            page += 1
            continue

        response.raise_for_status()
        data = response.json()

        if not data or "results" not in data or not data["results"]:
            print(f"No more data at page {page}.")
            break

        for therapist in data["results"]:
            name        = therapist.get("name", "N/A")
            hq          = therapist.get("headquarters", [{}])[0]
            location    = hq.get("population_name", "N/A")
            postal_code = hq.get("postal_code", "N/A")
            rating      = therapist.get("reviews_rating", "N/A")
            price       = therapist.get("service_price", "N/A")
            services    = therapist.get("top_services", "N/A")

            if isinstance(services, list):
                services = ", ".join(s["service_name"] for s in services)

            calendar_id = hq.get("schedule_calendar_id")
            availability = get_availability(calendar_id, "2025-01-13", 2) if calendar_id else ["N/A"]

            for av in availability:
                writer.writerow([name, location, postal_code, rating, price, av, services])

        print(f"Processed page {page}.")
        page += 1

print(f"Data saved to {output_file}.")

---

## Merge and Finalise the Dataset

Data was collected across two scraping sessions:
- **Session 1** (December 23, 2024): Parts 1–3 of the API pagination, saved as separate CSVs and merged into `merged_file.csv`.
- **Session 2** (January 13, 2025): A second full pass saved as `therapists_api.csv`.

We tag each record with its collection date and concatenate both sessions into the final dataset `therapists_final.csv`.

In [ ]:
# --- Session 1: merge the three partial files from December 23, 2024 ---
file1 = pd.read_csv("data/raw/1therapists_data.csv")
file2 = pd.read_csv("data/raw/2therapists_data.csv")
file3 = pd.read_csv("data/raw/3therapists_data.csv")

session1 = pd.concat([file1, file2, file3], axis=0)
session1["webscraping_date"] = pd.to_datetime("23/12/2024", format="%d/%m/%Y")
session1.to_csv("data/raw/session1_merged.csv", index=False)
print(f"Session 1: {len(session1):,} rows")

In [ ]:
# --- Session 2: tag the January 13, 2025 file ---
session2 = pd.read_csv("data/raw/therapists_api.csv")
session2["webscraping_date"] = pd.to_datetime("13/01/2025", format="%d/%m/%Y")
session2.to_csv("data/raw/session2_tagged.csv", index=False)
print(f"Session 2: {len(session2):,} rows")

In [ ]:
# --- Final merge ---
therapists_final = pd.concat([session1, session2], axis=0)
therapists_final.reset_index(drop=True, inplace=True)

os.makedirs("data/processed", exist_ok=True)
therapists_final.to_csv("data/processed/therapists_final.csv", index=False)

print(f"Final dataset: {len(therapists_final):,} rows, {therapists_final['Name'].nunique():,} unique therapists")
therapists_final.head()